<a href="https://colab.research.google.com/github/brandywii/dsa405-project/blob/main/DSA405_002_FA26_A9_bdwilki2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9: How the Web Is Built, and Scraping I

**DSA 405 · Week 9**

| | |
|---|---|
| **In class** | Friday, Oct 16 |
| **A9 due** | Thursday, Oct 22, 11:59 PM |
| **Target** | `books.toscrape.com`, a scraping sandbox |
| **Time** | ~25 min in class, ~60 min at home |

## Overview

A **scraper** is a program that downloads web pages and extracts data from them. Every
first scraper has the same three steps, and the steps do not change as scrapers get
bigger:

1. **Fetch** the page with `requests` (a Python library that downloads a web page the
   way a browser does)
2. **Find** the elements you want with a CSS selector (a short text pattern that names
   which HTML elements to pick out)
3. **Extract** the text and attributes from those elements into a DataFrame

Step 2 is the hard part. The tool for it is your browser, not Python - you find the
pattern by looking at the page's HTML, and only then write code that uses it.

We practice on `books.toscrape.com`, a **sandbox**: a practice site built for people who
are learning to scrape. It has no terms of service and no rate limits (a rate limit is a
cap on how many requests a site accepts from you in a given amount of time). Because we
practice on a sandbox this week, Week 10's work on a real site will be a debugging
problem, not an ethics problem.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, time, io
from bs4 import BeautifulSoup

# Tell the site who you are. Many sites refuse requests from scripts that do not identify themselves.
HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; teaching exercise)"}

pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| bs4 ready")

pandas 2.2.3 | bs4 ready


---
# Part 1: Explore (in class)

## Task 1.1: Fetch a page and look at what came back

`requests.get` downloads a page and returns a **response object** - a Python object that
holds the status code, the headers, and the text of the page. **Parsing** means turning
that raw text into a structure your code can search. Before parsing anything, look at
what actually came back. It is not always the page you expected.

In [ ]:
URL = "https://books.toscrape.com/catalogue/page-1.html"

r = requests.get(URL, headers=HEADERS, timeout=20)

print("status      :", r.status_code)
print("content-type:", r.headers.get("content-type"))
print("length      :", len(r.text), "characters")
print()
print(r.text[:600])

status      : 200
content-type: text/html
length      : 50469 characters



<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:30" />
        <meta name="description" c


Check the status code before you parse. A 404 ("page not found") error page still
returns text, and BeautifulSoup will parse that text without any error and find nothing
in it. Tip: `raise_for_status()` does this check for you in one line.

In [ ]:
r.raise_for_status()      # a 404 error page still returns text, so check the status code first
soup = BeautifulSoup(r.text, "html.parser")
print(soup.title.text.strip())

All products | Books to Scrape - Sandbox


## Task 1.2: Find selectors in the browser, not in Python

Do this part in your browser, not in this notebook. Open the page, right-click a book
title, choose **Inspect**, and read the HTML around it. (The Inspect panel shows you the
exact HTML the browser received, which is the same text `requests` received.)

There are three questions to answer:

1. What element wraps **one whole book**? That element is the **repeating unit**: the
   element that appears once for each book on the page.
2. Inside it, which element contains the **title**, which contains the **price**, and
   which contains the **availability**?
3. Is the title in the element's text, or in an **attribute** like `title` or `href`?
   (An attribute is a named value written inside the opening tag, such as `href="..."`.)

Write your findings down in the next cell before you write any code.

**Selectors found with Inspect:**

- One book: `...`
- Title: `...`
- Price: `...`
- Availability: `...`
- Link to detail page: `...`

Now test them. `soup.select()` takes a CSS selector and returns a list of every element
that matches; `soup.select_one()` returns only the first match.

In [ ]:
books = soup.select("article.product_pod")      # your repeating-unit selector

print("elements found:", len(books))
print()
if books:
    print(books[0].prettify()[:900])
else:
    print("Nothing matched. Go back to Inspect; do not guess.")

elements found: 20

<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



A correct repeating-unit selector matches exactly 20 elements, because each catalogue
page shows 20 books. Knowing the expected count *before* you look at the result is your
first verification check, and it is one you will reuse every time you scrape anything:
(1) decide what number you expect, (2) run the code, and (3) compare the two.

## Task 1.3: Extract one book, then all twenty

Get one book working completely before you write the loop. Debugging inside a loop is
harder than debugging a single case, because the error message does not tell you which
of the 20 books caused it.

In [ ]:
one = books[0]

# fill these in using the selectors you found
title = one.select_one("h3 a")["title"]
price = one.select_one("p.price_color").text
avail = one.select_one("p.instock.availability").text.strip()
link = one.select_one("h3 a")["href"]

#TO DO: Price fix -> Look back at Lab 4 to fix the Mokebaki char

print("title:", title)
print("price:", price)
print("avail:", avail)
print("link :", link)

title: A Light in the Attic
price: Â£51.77
avail: In stock
link : a-light-in-the-attic_1000/index.html


In [ ]:
rows = []
for b in books:
    rows.append({
        "title": b.select_one("h3 a")["title"],
        "price_raw": b.select_one("p.price_color").text,
        "availability_raw": b.select_one("p.instock.availability").text.strip(),
        "url": b.select_one("h3 a")["href"]})


page1 = pd.DataFrame(rows)
print(page1.shape)
page1.head()

(20, 4)


,title,price_raw,availability_raw,url
0,A Light in the Attic,Â£51.77,In stock,a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,Â£53.74,In stock,tipping-the-velvet_999/index.html
2,Soumission,Â£50.10,In stock,soumission_998/index.html
3,Sharp Objects,Â£47.82,In stock,sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,sapiens-a-brief-history-of-humankind_996/index...


Verify the result. Copy these three lines into the next cell and run them:

```python
assert len(page1) == 20, f"expected 20 books, got {len(page1)}"
assert page1.title.notna().all(), "some titles came back empty"
assert page1.title.nunique() == 20, "duplicate titles: are you re-reading the same element?"
```

The third one catches a mistake that produces no error message, and that almost everyone
makes once: calling `select_one` on `soup` (the whole page) inside the loop instead of on
`b` (the current book). That mistake returns the same element on every pass through the
loop, so every row gets the first book's value, and Python shows no warning at all. The
`nunique()` assert is the one-line check that catches it.

In [ ]:
# your assertions
assert len(page1) == 20, f"expected 20 books, got {len(page1)}"
assert page1.title.notna().all(), "some titles came back empty"
assert page1.title.nunique() == 20, "duplicate titles: are you re-reading the same element?"

---
## Checkpoint: submit before leaving

1. What selector matched one whole book, and how did you find it?
2. Was any field stored in an **attribute** rather than in the element's text? Which
   one, and how did you extract it?
3. Did all three assertions pass? If one failed, what was wrong?
4. Name one thing about this page's HTML that your scraper depends on, so that if the
   site changed it, your scraper would stop working.

*Answers here.*
1. `article.product_pod` was the selector that mtched one whole book. It can be found by visiting the provided url and clicking F12 to access the developer tools. There you will find the webpage html code/tags.

2. Yes, I saw two fields stored in an attribute: title was stored inside the `title` attribute and the url was stored in the `href`attribute.

3. Yes, all three assertions passed.

4. Scraper depends on `product_pod` in the `<article class>` tag. If it was changed, the scraper would no longer work.

---
# Part 2: A9, a First Scraper

Four tasks. Together they are the take-home half of the week (about 60 minutes).

## Task 2.1: Clean the scraped fields

Scraped data always arrives as text, and the text always has small problems. There are
three problems here:

- `price_raw` is a currency string, and there may be an unexpected extra character in
  front of the `£`
- The rating is stored in a **class name** (the `class` attribute of the rating
  element), not in any text
- `availability_raw` looks like useful data but is not. Print its unique values. The
  listing page only ever says "In stock". The actual stock count is on the detail pages,
  and you will collect it in Task 2.2.

Turn the first two into usable columns: `price` as a float and `rating` as an integer
from 1 to 5. Then write one sentence: what does `availability_raw` actually contain, and
what did it appear to contain?

For the rating, you need the `books` elements from Task 1.2 again, because `page1` does
not contain the class names. The list of classes on the rating element contains a number
word, such as `Three`. Map that word to its number with the dictionary in the next cell.

In [ ]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

# extract rating from the class attribute, then clean price and availability

**The encoding problem.** Print `repr(page1.price_raw.iloc[0])`. Any strange character
in front of the `£` is a character-encoding problem of the kind we handled in Week 4
(text that was written in one encoding and read back in another). Diagnose it and state
what you did about it. Either approach is fine: repair the encoding, or extract the
number and explain why you ignored the rest.

*Answer here.*

## Task 2.2: Follow a link to a detail page

Listing pages rarely contain everything you need, so real scraping means following links
to other pages.

Pick **five** books from the DataFrame, fetch each book's detail page, and extract three
fields that exist only there: (1) the **UPC**, a product code; (2) the **product
description**; and (3) the stock count inside "In stock (22 available)". Extract the
number from that string; it is the `in_stock` column the listing page did not provide in
Task 2.1.

**Wait one second between requests**, even on the sandbox. The real sites you will want
for P3 block a client that sends too many requests too quickly, and if a site blocks your
address during the project weeks, you cannot collect your data. Add the one-second pause
now, so it is already a habit when you work with a real site.

In [ ]:
details = []
for url in page1.url.head(5):
    full = requests.compat.urljoin(URL, url)
    resp = requests.get(full, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    s = BeautifulSoup(resp.text, "html.parser")

    details.append({
        "url": url,
        "upc": ...,
        "description": ...,
    })
    time.sleep(1)          # wait 1 second so we do not overload the server

pd.DataFrame(details)

,url,upc,description
0,a-light-in-the-attic_1000/index.html,Ellipsis,Ellipsis
1,tipping-the-velvet_999/index.html,Ellipsis,Ellipsis
2,soumission_998/index.html,Ellipsis,Ellipsis
3,sharp-objects_997/index.html,Ellipsis,Ellipsis
4,sapiens-a-brief-history-of-humankind_996/index...,Ellipsis,Ellipsis


**Break it on purpose.** A few books in this catalogue have no description element at
all. Here is one of them:

`catalogue/the-bridge-to-consciousness-im-writing-the-bridge-between-science-and-our-old-and-new-beliefs_840/index.html`

Run your detail-page code on that page. Find out what the code does when the element is
missing, then change the code so it returns something sensible (for example `None`)
instead of either (1) crashing or (2) returning a different element with no error
message. The second outcome is worse, because you would record the wrong text and not
know it.

*What happened, and what was the fix?*

## Task 2.3: Build a reusable function

Wrap the page-scraping steps in a function. In Week 10 we write a **crawl** - a loop that
scrapes many pages - and that loop will call this function once per page.

Requirements:

- Takes a page number and returns a DataFrame
- Raises an error on a bad HTTP status
- **Raises an error if it does not find 20 books**, instead of returning a DataFrame
  with fewer rows
- Calls `time.sleep` before it returns
- Has a docstring (the description text at the top of a function) that states what the
  function expects to find on the page

In [ ]:
def scrape_page(n, expected=20):
    """Scrape one catalogue page of books.toscrape.com.

    Raises an error if the page is unreachable or does not contain `expected`
    books. A page with fewer books than expected means the site has changed,
    and a wrong result with no error message is worse than a crash.
    """
    ...


test = scrape_page(1)
print(test.shape)
test.head(3)

AttributeError: 'NoneType' object has no attribute 'shape'

## Task 2.4: What happens if the website changes?

The webpage is outside your control. Your scraper depends on choices someone else made (the web developer) and
that person can change those choices without telling you.


**Describe how changes could be detected** if nobody told you about it. Describe
   a check that could run once a week that would check if anything about the website has changed.

*Answer here.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A9_[yourUnityID].ipynb`
4. Upload to the **A9** space on Moodle.

The **Checkpoint** section goes separately to **Week 9 In-Class Activity** before
leaving class on Friday. A9 is due **Thursday, Oct 22, 11:59 PM**.